# 9 · OceanLab: ¿qué ola habrá dentro de 10 minutos?

En OceanLab el tiempo está **acoplado**: las borrascas bajan la presión, el viento sale de la diferencia de
presión y el oleaje crece con el viento… **con horas de retraso**. A la escala del simulador (1 s real = 1 min),
10 minutos de clase son 10 horas de mar.

Vamos a prever la altura de ola de cada boya dentro de 10 minutos y a compararla con la previsión más simple
que existe, la **persistencia**: «dentro de 10 minutos, la misma ola que ahora».

Datos: el conjunto `oceanlab` (una muestra de cada boya cada 30 s).

In [ ]:
from labia import datos, guardar
import numpy as np, pandas as pd, matplotlib.pyplot as plt

df = datos("oceanlab").sort_values(["boya", "ts"])
print(len(df), "muestras de", df.boya.nunique(), "boyas en", round((df.ts.max() - df.ts.min()).total_seconds() / 60), "minutos")
df.groupby("boya").ola.max().sort_values(ascending=False).head()

## Una boya en una borrasca

Elige la boya que ha tenido la ola más alta y mira cómo se mueven juntas la presión, el viento y la ola.

In [ ]:
boya = df.groupby("boya").ola.max().idxmax()
b = df[df.boya == boya]
# el tramo más largo sin huecos (si la recogida se paró, no se dibuja una recta encima del hueco)
tramos = (b.ts.diff() > pd.Timedelta("5min")).cumsum()
b = b[tramos == tramos.value_counts().idxmax()]
fig, ejes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
ejes[0].plot(b.ts, b.presion, color="tab:purple"); ejes[0].set_ylabel("presión (hPa)")
ejes[1].plot(b.ts, b.viento, color="tab:green"); ejes[1].set_ylabel("viento (m/s)")
ejes[2].plot(b.ts, b.ola, color="tab:blue"); ejes[2].set_ylabel("ola (m)")
for ax in ejes: ax.grid(alpha=.3)
ejes[0].set_title(f"{boya}: la presión baja, el viento sube… y la ola llega después");

## Preparar los datos: el futuro y el pasado de cada muestra

Para cada muestra buscamos, **en la misma boya**:

* la ola **10 minutos después** (lo que queremos prever: `ola_futura`);
* la presión y el viento **10 minutos antes**, para saber si están cambiando (`dp`, `dv`).

`merge_asof` busca la muestra más cercana en el tiempo (con 1 minuto de tolerancia). Node-RED hace lo mismo en
la *function* «Ventana de 10 min».

In [ ]:
H = pd.Timedelta("10min"); TOL = pd.Timedelta("1min")
base = df[["ts", "boya", "ola", "viento", "presion"]].sort_values("ts")

futuro = base[["ts", "boya", "ola"]].rename(columns={"ola": "ola_futura"})
futuro["ts"] = futuro.ts - H                       # la muestra de dentro de 10 min, puesta en «ahora»
pasado = base[["ts", "boya", "presion", "viento"]].rename(columns={"presion": "p_antes", "viento": "v_antes"})
pasado["ts"] = pasado.ts + H                       # la de hace 10 min, puesta en «ahora»

d = pd.merge_asof(base, futuro.sort_values("ts"), on="ts", by="boya", tolerance=TOL, direction="nearest")
d = pd.merge_asof(d, pasado.sort_values("ts"), on="ts", by="boya", tolerance=TOL, direction="nearest")
d["dp"] = d.presion - d.p_antes
d["dv"] = d.viento - d.v_antes
d = d.dropna(subset=["ola_futura", "dp", "dv"])
print(len(d), "muestras con pasado y futuro")
d[["boya", "ola", "viento", "presion", "dp", "dv", "ola_futura"]].head()

## Persistencia frente a modelos

Entrenamos con el primer 60 % del tiempo y probamos con el 40 % final (no se puede barajar: el mar de las 10:05 se
parece mucho al de las 10:04).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

ENTRADAS = ["ola", "viento", "presion", "dp", "dv"]
corte = d.ts.quantile(0.6)
ent, prueba = d[d.ts <= corte], d[d.ts > corte]

resultados = {"persistencia": mean_absolute_error(prueba.ola_futura, prueba.ola)}
lineal = LinearRegression().fit(ent[ENTRADAS], ent.ola_futura)
bosque = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=0).fit(ent[ENTRADAS], ent.ola_futura)
for nombre, m in [("lineal", lineal), ("bosque aleatorio", bosque)]:
    resultados[nombre] = mean_absolute_error(prueba.ola_futura, m.predict(prueba[ENTRADAS]))
pd.Series(resultados).round(3).to_frame("error medio (m)")

La persistencia es **difícil de batir**: casi siempre el mar de dentro de un rato se parece al de ahora. ¿Dónde
gana el modelo? Cuando el mar **cambia**:

In [ ]:
cambia = (prueba.ola_futura - prueba.ola).abs() > 0.5
tabla = pd.DataFrame({
    "persistencia": [mean_absolute_error(prueba.ola_futura[c], prueba.ola[c]) for c in (~cambia, cambia)],
    "lineal": [mean_absolute_error(prueba.ola_futura[c], lineal.predict(prueba[ENTRADAS][c])) for c in (~cambia, cambia)],
    "bosque": [mean_absolute_error(prueba.ola_futura[c], bosque.predict(prueba[ENTRADAS][c])) for c in (~cambia, cambia)],
}, index=["el mar casi no cambia", "la ola cambia más de 0,5 m"]).round(3)
tabla["muestras"] = [(~cambia).sum(), cambia.sum()]
tabla

In [ ]:
pb = prueba[prueba.boya == boya]
plt.figure(figsize=(11, 4))
plt.plot(pb.ts + H, pb.ola_futura, "k", lw=2, label="lo que pasó")
plt.plot(pb.ts + H, pb.ola, "--", label="persistencia (la ola de hace 10 min)")
plt.plot(pb.ts + H, bosque.predict(pb[ENTRADAS]), label="bosque aleatorio (previsto hace 10 min)")
plt.ylabel("ola (m)"); plt.grid(alpha=.3); plt.legend(); plt.title(f"{boya}: previsión a 10 minutos");

In [ ]:
pd.Series(lineal.coef_, index=ENTRADAS).round(4).to_frame("coeficiente")

> Lee los coeficientes: ¿cuánto pesa la ola de ahora? ¿Y el viento? Un `dp` negativo (la presión está bajando)
> ¿sube o baja la previsión? Es lo que hace un meteorólogo con el barómetro.

Publica el modelo que mejor te haya ido con los datos de prueba (aquí, el bosque aleatorio). Node-RED lo usará con
cada muestra de cada boya.

In [ ]:
mejor = min(["lineal", "bosque aleatorio"], key=resultados.get)
modelo = {"lineal": lineal, "bosque aleatorio": bosque}[mejor]
print("se publica:", mejor)
modelo.fit(d[ENTRADAS], d.ola_futura)             # para publicar, con todas las muestras
guardar(modelo, "oceanlab-oleaje", entradas=ENTRADAS, tipo="regresion", salida="ola dentro de 10 min",
        descripcion="Previsión de la altura de ola a 10 minutos (10 h de mar) con la ola, el viento, la presión y su tendencia",
        metricas={"mae_m": round(float(resultados[mejor]), 3),
                  "mae_persistencia_m": round(float(resultados["persistencia"]), 3)})

## ✏️ Tu turno

1. Añade `ola_eq = 0.0214 * viento**2` (la ola de un mar totalmente desarrollado para ese viento) a las entradas.
   ¿Mejora? ¿Por qué un poco de física ayuda a un modelo lineal?
2. Cambia el horizonte a 5 y a 20 minutos. ¿Cómo cambia el error de la persistencia? ¿Y la ventaja del modelo?
3. ¿Sirve el mismo modelo para todas las boyas? Entrena uno solo con boyas tropicales (`abs(latitud) < 25`) y
   pruébalo en las de latitudes altas.
4. El modelo no sabe **dónde** está la borrasca, solo lo que mide su boya. ¿Qué información de las boyas
   **vecinas** añadirías para prever mejor la llegada de una tormenta?